In [ ]:
!pip install -q --upgrade pip
!pip install -q --upgrade torch torchvision transformers datasets timm pillow scikit-learn
!pip install -q peft accelerate bitsandbytes
!pip install --upgrade --force-reinstall huggingface_hub -q
!pip install optuna
!pip install git+https://github.com/openai/CLIP.git
!pip install ftfy regex tqdm

In [ ]:
!pip install -q autogluon

In [ ]:
# ==================== OPTIMIZED AUTOGLUON INSTALLATION ====================
# This reduces CPU load by installing in stages

print("📦 Stage 1: Installing core dependencies...")
!pip install -q --no-cache-dir numpy pandas scikit-learn

print("📦 Stage 2: Installing ML frameworks...")
!pip install -q --no-cache-dir lightgbm xgboost

print("📦 Stage 3: Installing AutoGluon (this takes longest)...")
!pip install --no-cache-dir autogluon.tabular

print("✅ All installations complete!")

# Verify
from autogluon.tabular import TabularPredictor
print("✅ AutoGluon ready!")

In [ ]:
# Check if transformers installed correctly
import transformers
import torch
import accelerate
from PIL import Image

print(f"✅ transformers: {transformers.__version__}")
print(f"✅ torch: {torch.__version__}")
print(f"✅ accelerate: {accelerate.__version__}")
print(f"✅ PIL (pillow): {Image.__version__ if hasattr(Image, '__version__') else 'Installed'}")
print("\n🎉 All packages installed successfully!")

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Add your HF token in Kaggle Secrets first (Add-ons -> Secrets -> Add: HF_TOKEN)
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
!ls /kaggle/input

In [ ]:
import os
import pandas as pd

# Verify paths
base_path = '/kaggle/input/ml-challenge-2-2025/'
print(f"Train CSV: {os.path.exists(base_path + 'train.csv')}")
print(f"Test CSV: {os.path.exists(base_path + 'test.csv')}")
print(f"Images: {os.path.exists(base_path + 'images/')}")

# Check data
train_df = pd.read_csv(base_path + 'train.csv')
test_df = pd.read_csv(base_path + 'test.csv')
print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")

# Count images
images = [f for f in os.listdir(base_path + 'images/') if f.endswith(('.jpg', '.png'))]
print(f"\nTotal images: {len(images)}")
print(f"Sample images: {images[:3]}")

In [ ]:
!pip install nltk

In [ ]:
#KISHORE TAKE THIS CODE and remove all the bs, see if u can make it even more feature rich and run
#PS: TRY ADDING IMAGES BRO WTF
"""
AUTOGLUON + ULTRA-RICH FEATURE ENGINEERING PIPELINE
========================================================================
Strategy: Extract 50+ features from catalog_content text alone
Target: <48% SMAPE through intelligent text parsing and NLP features
========================================================================
"""

import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')
import glob
import os
from autogluon.tabular import TabularPredictor
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer

# Download required NLTK data (run once)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('vader_lexicon')
except LookupError:
    nltk.download('vader_lexicon')

# ==================== CONFIGURATION ====================
CONFIG = {
    'train_csv': '/kaggle/input/ml-challenge-2-2025/train.csv',
    'test_csv': '/kaggle/input/ml-challenge-2-2025/test.csv',
    'previous_outputs_dir': '/kaggle/input/my-previous-price-predictions',
    'output_path': '/kaggle/working/test_out.csv',
    'autogluon_path': './autogluon_models',
    
    # AutoGluon settings
    'time_limit': 3600,  # 1 hour
    'preset': 'best_quality',
    'num_bag_folds': 10,  # More folds for stability
    
    'random_state': 42,
    
    # Historical scores for meta-learning
    'historical_scores': {
        'output_49.385.csv': 49.38488936456298,
        'output_50.314.csv': 50.31381129428071,
        'output_51.375.csv': 51.3752876230568,
        'output_60.531.csv': 60.53146486141669,
        'output_61.586.csv': 61.58578507849365,
    }
}

# ==================== SMAPE METRIC ====================
def calculate_smape(actual, predicted):
    """Symmetric Mean Absolute Percentage Error"""
    actual = np.array(actual)
    predicted = np.array(predicted)
    predicted = np.maximum(predicted, 0.01)
    
    numerator = np.abs(predicted - actual)
    denominator = (np.abs(actual) + np.abs(predicted)) / 2
    smape = np.mean(numerator / (denominator + 1e-10)) * 100
    
    return smape

# ==================== ULTRA-RICH FEATURE ENGINEERING ====================

def extract_pack_quantity(text):
    """Extract pack quantity with comprehensive patterns"""
    if pd.isna(text):
        return 1
    text_str = str(text).lower()
    
    # Multiple pack patterns (ordered by priority)
    patterns = [
        # Direct value fields
        (r'value:\s*(\d+\.?\d*)', 1),
        (r'quantity:\s*(\d+\.?\d*)', 1),
        (r'count:\s*(\d+\.?\d*)', 1),
        
        # Pack patterns
        (r'\(pack of (\d+)\)', 1),
        (r'pack of (\d+)', 1),
        (r'\((\d+) pack\)', 1),
        (r'(\d+)\s*pack\b', 1),
        (r'(\d+)-pack\b', 1),
        (r'(\d+)\s*count\b', 1),
        (r'set of (\d+)', 1),
        (r'(\d+)\s*piece', 1),
        (r'(\d+)\s*ct\b', 1),
        (r'(\d+)\s*units?', 1),
        (r'(\d+)\s*items?', 1),
        
        # Multiplier patterns
        (r'(\d+)\s*x\s*\d+', 1),  # 6x24 pattern
        (r'(\d+\.?\d*)\s*dozen', 12),  # dozens
        (r'(\d+\.?\d*)\s*gross', 144),  # gross
    ]
    
    for pattern, multiplier in patterns:
        match = re.search(pattern, text_str)
        if match:
            qty = float(match.group(1))
            total_qty = qty * multiplier
            if 1 <= total_qty <= 1000:  # Reasonable range
                return total_qty
    
    return 1

def extract_unit_size_and_type(text):
    """Extract unit size with comprehensive measurement patterns"""
    if pd.isna(text):
        return 0, 'unknown'
    
    text_str = str(text).lower()
    
    # Weight patterns (convert to grams)
    weight_patterns = [
        (r'(\d+\.?\d*)\s*(?:grams?|gms?|g)\b', 1),
        (r'(\d+\.?\d*)\s*(?:kilograms?|kgs?|kg)\b', 1000),
        (r'(\d+\.?\d*)\s*(?:pounds?|lbs?|lb)\b', 453.592),
        (r'(\d+\.?\d*)\s*(?:ounces?|ozs?|oz)\b', 28.3495),
    ]
    
    # Volume patterns (convert to ml)
    volume_patterns = [
        (r'(\d+\.?\d*)\s*(?:milliliters?|mls?|ml)\b', 1),
        (r'(\d+\.?\d*)\s*(?:liters?|ls?|l)\b', 1000),
        (r'(\d+\.?\d*)\s*(?:fluid ounces?|fl ?ozs?|fl ?oz)\b', 29.5735),
        (r'(\d+\.?\d*)\s*(?:cups?)\b', 236.588),
        (r'(\d+\.?\d*)\s*(?:pints?|pts?|pt)\b', 473.176),
        (r'(\d+\.?\d*)\s*(?:quarts?|qts?|qt)\b', 946.353),
        (r'(\d+\.?\d*)\s*(?:gallons?|gals?|gal)\b', 3785.41),
    ]
    
    # Length patterns (convert to cm)
    length_patterns = [
        (r'(\d+\.?\d*)\s*(?:centimeters?|cms?|cm)\b', 1),
        (r'(\d+\.?\d*)\s*(?:meters?|ms?|m)\b', 100),
        (r'(\d+\.?\d*)\s*(?:inches?|ins?|in|")\b', 2.54),
        (r'(\d+\.?\d*)\s*(?:feet|ft|\')\b', 30.48),
        (r'(\d+\.?\d*)\s*(?:millimeters?|mms?|mm)\b', 0.1),
    ]
    
    # Check weight first
    for pattern, conversion in weight_patterns:
        match = re.search(pattern, text_str)
        if match:
            size = float(match.group(1)) * conversion
            return size, 'weight'
    
    # Check volume
    for pattern, conversion in volume_patterns:
        match = re.search(pattern, text_str)
        if match:
            size = float(match.group(1)) * conversion
            return size, 'volume'
    
    # Check length
    for pattern, conversion in length_patterns:
        match = re.search(pattern, text_str)
        if match:
            size = float(match.group(1)) * conversion
            return size, 'length'
    
    return 0, 'unknown'

def extract_price_indicators(text):
    """Extract price-relevant indicators from text"""
    if pd.isna(text):
        return {}
    
    text_str = str(text).lower()
    
    indicators = {}
    
    # Direct price patterns
    price_patterns = [
        r'\$(\d+\.?\d*)',
        r'(\d+\.?\d*)\s*dollars?',
        r'price[:\s]*\$?(\d+\.?\d*)',
        r'cost[:\s]*\$?(\d+\.?\d*)',
        r'msrp[:\s]*\$?(\d+\.?\d*)',
    ]
    
    found_prices = []
    for pattern in price_patterns:
        matches = re.findall(pattern, text_str)
        found_prices.extend([float(p) for p in matches if float(p) < 10000])
    
    indicators['has_price_mention'] = len(found_prices) > 0
    indicators['min_mentioned_price'] = min(found_prices) if found_prices else 0
    indicators['max_mentioned_price'] = max(found_prices) if found_prices else 0
    indicators['avg_mentioned_price'] = np.mean(found_prices) if found_prices else 0
    
    # Value indicators
    value_keywords = [
        'value', 'affordable', 'budget', 'cheap', 'economy', 'discount',
        'sale', 'deal', 'bargain', 'low cost', 'inexpensive'
    ]
    indicators['value_score'] = sum(1 for kw in value_keywords if kw in text_str)
    
    # Premium indicators
    premium_keywords = [
        'premium', 'luxury', 'high-end', 'expensive', 'deluxe', 'gourmet',
        'artisan', 'professional', 'commercial grade', 'top quality'
    ]
    indicators['premium_score'] = sum(1 for kw in premium_keywords if kw in text_str)
    
    return indicators

def extract_brand_and_quality_features(text):
    """Extract brand and quality indicators"""
    if pd.isna(text):
        return {}
    
    text_str = str(text).lower()
    features = {}
    
    # Major brand indicators
    major_brands = [
        'amazon', 'amazonbasics', 'kirkland', 'great value', 'equate',
        'nike', 'adidas', 'apple', 'samsung', 'sony', 'lg', 'hp', 'dell',
        'starbucks', 'nescafe', 'keurig', 'folgers', 'dunkin', 'coca cola',
        'pepsi', 'nestle', 'kraft', 'general mills', 'kelloggs'
    ]
    
    features['major_brand_count'] = sum(1 for brand in major_brands if brand in text_str)
    features['has_major_brand'] = features['major_brand_count'] > 0
    
    # Quality certifications
    certifications = [
        'organic', 'non-gmo', 'gluten-free', 'vegan', 'kosher', 'halal',
        'fair trade', 'sustainable', 'eco-friendly', 'bpa-free', 'natural',
        'certified', 'approved', 'tested', 'verified'
    ]
    
    features['certification_count'] = sum(1 for cert in certifications if cert in text_str)
    features['quality_score'] = features['certification_count']
    
    # Material quality indicators
    premium_materials = [
        'stainless steel', 'aluminum', 'titanium', 'carbon fiber', 'leather',
        'cotton', 'wool', 'silk', 'cashmere', 'bamboo', 'ceramic', 'glass',
        'hardwood', 'solid wood', 'mahogany', 'oak'
    ]
    
    features['premium_material_count'] = sum(1 for mat in premium_materials if mat in text_str)
    
    return features

def extract_category_features(text):
    """Extract product category indicators"""
    if pd.isna(text):
        return {}
    
    text_str = str(text).lower()
    features = {}
    
    # Category mappings
    categories = {
        'food': ['food', 'snack', 'meal', 'eat', 'edible', 'nutrition', 'ingredient',
                'flavor', 'taste', 'recipe', 'cooking', 'baking', 'seasoning'],
        
        'beverage': ['drink', 'beverage', 'juice', 'water', 'coffee', 'tea', 'soda',
                    'wine', 'beer', 'alcohol', 'smoothie', 'shake', 'latte'],
        
        'electronics': ['electronic', 'digital', 'battery', 'usb', 'cable', 'charger',
                       'wireless', 'bluetooth', 'smart', 'tech', 'device', 'gadget'],
        
        'health': ['health', 'vitamin', 'supplement', 'medical', 'wellness', 'fitness',
                  'pharmacy', 'medicine', 'therapeutic', 'healing', 'treatment'],
        
        'beauty': ['beauty', 'cosmetic', 'skin', 'hair', 'makeup', 'skincare',
                  'shampoo', 'conditioner', 'lotion', 'cream', 'serum'],
        
        'home': ['home', 'house', 'kitchen', 'bathroom', 'bedroom', 'living room',
                'furniture', 'decor', 'cleaning', 'laundry', 'storage'],
        
        'clothing': ['clothing', 'apparel', 'shirt', 'pants', 'dress', 'shoes',
                    'jacket', 'coat', 'underwear', 'socks', 'hat', 'accessories'],
        
        'sports': ['sports', 'fitness', 'exercise', 'workout', 'gym', 'athletic',
                  'running', 'swimming', 'cycling', 'yoga', 'training'],
        
        'automotive': ['car', 'auto', 'vehicle', 'automotive', 'tire', 'engine',
                      'brake', 'oil', 'filter', 'maintenance', 'repair'],
        
        'baby': ['baby', 'infant', 'toddler', 'child', 'kids', 'nursery',
                'diaper', 'formula', 'toy', 'stroller', 'crib']
    }
    
    for category, keywords in categories.items():
        score = sum(1 for kw in keywords if kw in text_str)
        features[f'{category}_score'] = score
        features[f'is_{category}'] = score > 0
    
    # Find dominant category
    category_scores = {cat: features[f'{cat}_score'] for cat in categories.keys()}
    dominant_category = max(category_scores, key=category_scores.get)
    features['dominant_category'] = dominant_category
    features['category_confidence'] = category_scores[dominant_category]
    
    return features

def extract_text_statistics(text):
    """Extract comprehensive text statistics"""
    if pd.isna(text):
        return {}
    
    text_str = str(text)
    text_lower = text_str.lower()
    features = {}
    
    # Basic statistics
    features['text_length'] = len(text_str)
    features['word_count'] = len(text_str.split())
    features['char_count'] = len(text_str)
    features['sentence_count'] = len(re.findall(r'[.!?]+', text_str))
    
    # Character analysis
    features['uppercase_count'] = sum(1 for c in text_str if c.isupper())
    features['lowercase_count'] = sum(1 for c in text_str if c.islower())
    features['digit_count'] = sum(1 for c in text_str if c.isdigit())
    features['punctuation_count'] = sum(1 for c in text_str if c in string.punctuation)
    features['whitespace_count'] = sum(1 for c in text_str if c.isspace())
    
    # Ratios
    if features['char_count'] > 0:
        features['uppercase_ratio'] = features['uppercase_count'] / features['char_count']
        features['digit_ratio'] = features['digit_count'] / features['char_count']
        features['punctuation_ratio'] = features['punctuation_count'] / features['char_count']
    else:
        features['uppercase_ratio'] = features['digit_ratio'] = features['punctuation_ratio'] = 0
    
    if features['word_count'] > 0:
        features['avg_word_length'] = features['char_count'] / features['word_count']
    else:
        features['avg_word_length'] = 0
    
    # Special patterns
    features['bullet_count'] = text_str.count('•') + text_str.count('*') + text_str.count('-')
    features['newline_count'] = text_str.count('\n')
    features['url_count'] = len(re.findall(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', text_str))
    features['email_count'] = len(re.findall(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', text_str))
    features['phone_count'] = len(re.findall(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', text_str))
    
    # Word complexity
    words = text_lower.split()
    if words:
        features['unique_word_count'] = len(set(words))
        features['unique_word_ratio'] = features['unique_word_count'] / len(words)
        features['long_word_count'] = sum(1 for word in words if len(word) > 6)
        features['long_word_ratio'] = features['long_word_count'] / len(words)
    else:
        features['unique_word_count'] = features['unique_word_ratio'] = 0
        features['long_word_count'] = features['long_word_ratio'] = 0
    
    return features

def extract_number_features(text):
    """Extract all numerical information from text"""
    if pd.isna(text):
        return {}
    
    text_str = str(text)
    features = {}
    
    # Find all numbers
    numbers = re.findall(r'\d+\.?\d*', text_str)
    numeric_values = [float(x) for x in numbers if float(x) < 1000000]  # Filter outliers
    
    if numeric_values:
        features['number_count'] = len(numeric_values)
        features['min_number'] = min(numeric_values)
        features['max_number'] = max(numeric_values)
        features['sum_numbers'] = sum(numeric_values)
        features['avg_number'] = np.mean(numeric_values)
        features['std_number'] = np.std(numeric_values)
        features['median_number'] = np.median(numeric_values)
        
        # Number ranges
        features['small_numbers'] = sum(1 for x in numeric_values if x < 10)
        features['medium_numbers'] = sum(1 for x in numeric_values if 10 <= x < 100)
        features['large_numbers'] = sum(1 for x in numeric_values if x >= 100)
        
        # Specific patterns
        features['percentage_count'] = len(re.findall(r'\d+\.?\d*\s*%', text_str))
        features['decimal_count'] = sum(1 for x in numbers if '.' in x)
        features['year_mentions'] = sum(1 for x in numeric_values if 1900 <= x <= 2030)
        
    else:
        # Default values when no numbers found
        num_features = ['number_count', 'min_number', 'max_number', 'sum_numbers',
                       'avg_number', 'std_number', 'median_number', 'small_numbers',
                       'medium_numbers', 'large_numbers', 'percentage_count',
                       'decimal_count', 'year_mentions']
        for feat in num_features:
            features[feat] = 0
    
    return features

def extract_sentiment_features(text):
    """Extract sentiment and emotional indicators"""
    if pd.isna(text):
        return {}
    
    text_str = str(text).lower()
    features = {}
    
    # Initialize sentiment analyzer
    sia = SentimentIntensityAnalyzer()
    sentiment_scores = sia.polarity_scores(text_str)
    
    features['sentiment_compound'] = sentiment_scores['compound']
    features['sentiment_positive'] = sentiment_scores['pos']
    features['sentiment_negative'] = sentiment_scores['neg']
    features['sentiment_neutral'] = sentiment_scores['neu']
    
    # Emotional keywords
    positive_words = ['excellent', 'amazing', 'great', 'fantastic', 'wonderful',
                     'perfect', 'best', 'superior', 'outstanding', 'exceptional']
    negative_words = ['bad', 'terrible', 'awful', 'poor', 'worst', 'horrible',
                     'disappointing', 'defective', 'broken', 'useless']
    
    features['positive_word_count'] = sum(1 for word in positive_words if word in text_str)
    features['negative_word_count'] = sum(1 for word in negative_words if word in text_str)
    
    # Intensity words
    intensity_words = ['very', 'extremely', 'super', 'ultra', 'mega', 'hyper',
                      'incredibly', 'amazingly', 'exceptionally', 'remarkably']
    features['intensity_word_count'] = sum(1 for word in intensity_words if word in text_str)
    
    return features

def create_ultra_rich_features(df):
    """Create comprehensive feature set from catalog_content"""
    print("🔧 Engineering ultra-rich features from text...")
    
    text_series = df['catalog_content'].fillna('')
    features_df = pd.DataFrame()
    
    print("   ├── Extracting pack quantities...")
    features_df['pack_qty'] = text_series.apply(extract_pack_quantity)
    
    print("   ├── Extracting unit sizes and types...")
    size_type_data = text_series.apply(extract_unit_size_and_type)
    features_df['unit_size'] = [x[0] for x in size_type_data]
    features_df['unit_type'] = [x[1] for x in size_type_data]
    
    # Create unit type dummies
    unit_type_dummies = pd.get_dummies(features_df['unit_type'], prefix='unit_type')
    features_df = pd.concat([features_df, unit_type_dummies], axis=1)
    features_df.drop('unit_type', axis=1, inplace=True)
    
    print("   ├── Extracting price indicators...")
    price_features = text_series.apply(extract_price_indicators)
    price_df = pd.DataFrame(price_features.tolist())
    features_df = pd.concat([features_df, price_df], axis=1)
    
    print("   ├── Extracting brand and quality features...")
    brand_features = text_series.apply(extract_brand_and_quality_features)
    brand_df = pd.DataFrame(brand_features.tolist())
    features_df = pd.concat([features_df, brand_df], axis=1)
    
    print("   ├── Extracting category features...")
    category_features = text_series.apply(extract_category_features)
    category_df = pd.DataFrame(category_features.tolist())
    features_df = pd.concat([features_df, category_df], axis=1)
    
    print("   ├── Extracting text statistics...")
    text_features = text_series.apply(extract_text_statistics)
    text_df = pd.DataFrame(text_features.tolist())
    features_df = pd.concat([features_df, text_df], axis=1)
    
    print("   ├── Extracting number features...")
    number_features = text_series.apply(extract_number_features)
    number_df = pd.DataFrame(number_features.tolist())
    features_df = pd.concat([features_df, number_df], axis=1)
    
    print("   ├── Extracting sentiment features...")
    sentiment_features = text_series.apply(extract_sentiment_features)
    sentiment_df = pd.DataFrame(sentiment_features.tolist())
    features_df = pd.concat([features_df, sentiment_df], axis=1)
    
    # Create interaction features
    print("   ├── Creating interaction features...")
    features_df['total_volume'] = features_df['pack_qty'] * features_df['unit_size']
    features_df['volume_per_pack'] = features_df['unit_size'] / (features_df['pack_qty'] + 1)
    features_df['quality_density'] = features_df['quality_score'] / (features_df['total_volume'] + 1)
    features_df['brand_quality_score'] = features_df['major_brand_count'] * features_df['quality_score']
    features_df['premium_value_ratio'] = (features_df['premium_score'] + 1) / (features_df['value_score'] + 1)
    features_df['text_complexity'] = features_df['unique_word_ratio'] * features_df['avg_word_length']
    features_df['number_density'] = features_df['number_count'] / (features_df['word_count'] + 1)
    
    # Log transforms for skewed features
    print("   ├── Creating log transforms...")
    log_features = ['pack_qty', 'unit_size', 'total_volume', 'text_length', 
                   'word_count', 'number_count', 'sum_numbers']
    for feat in log_features:
        if feat in features_df.columns:
            features_df[f'log_{feat}'] = np.log1p(features_df[feat])
    
    # Binned features
    print("   ├── Creating binned features...")
    features_df['pack_qty_bin'] = pd.cut(features_df['pack_qty'], 
                                        bins=[0, 1, 2, 5, 10, float('inf')], 
                                        labels=[0, 1, 2, 3, 4]).astype(float)
    
    features_df['text_length_bin'] = pd.cut(features_df['text_length'], 
                                           bins=[0, 100, 500, 1000, 2000, float('inf')], 
                                           labels=[0, 1, 2, 3, 4]).astype(float)
    
    # Keep original text for AutoGluon's built-in NLP
    features_df['catalog_content'] = df['catalog_content'].fillna('')
    
    # Fill any remaining NaN values
    features_df = features_df.fillna(0)
    
    print(f"✓ Created {features_df.shape[1]} features from catalog_content")
    print(f"   └── Feature types: numerical ({len([c for c in features_df.columns if features_df[c].dtype != 'object'])}), "
          f"text (1)")
    
    return features_df

# ==================== LOAD HISTORICAL PREDICTIONS ====================
def load_historical_with_intelligence(test_ids):
    """Intelligently load and weight historical predictions"""
    print("\n" + "="*80)
    print("📚 LOADING HISTORICAL PREDICTIONS (META-LAYER)")
    print("="*80)
    
    if CONFIG['previous_outputs_dir'] is None or not os.path.exists(CONFIG['previous_outputs_dir']):
        print("ℹ️  No historical data directory found")
        return None, None, None
    
    csv_files = glob.glob(os.path.join(CONFIG['previous_outputs_dir'], '*.csv'))
    
    if not csv_files:
        print("ℹ️  No CSV files in historical directory")
        return None, None, None
    
    print(f"Found {len(csv_files)} historical submission(s)")
    
    historical_preds = {}
    scores = {}
    
    # Load and validate historical predictions
    for csv_file in csv_files:
        filename = os.path.basename(csv_file)
        
        # Extract SMAPE from filename
        score_match = re.search(r'(\d+\.\d+)', filename)
        if score_match:
            score = float(score_match.group(1))
        else:
            score = None
            for key, val in CONFIG['historical_scores'].items():
                if key in filename or filename in key:
                    score = val
                    break
        
        if score is None:
            print(f"⚠️  Skipping {filename} (can't determine SMAPE)")
            continue
        
        try:
            df = pd.read_csv(csv_file)
            
            # Find price column
            price_col = None
            for col in df.columns:
                if any(keyword in col.lower() for keyword in ['price', 'pred', 'target']):
                    price_col = col
                    break
            
            if price_col is None and len(df.columns) >= 2:
                price_col = df.columns[1]
            
            if price_col is None:
                print(f"⚠️  Skipping {filename} (no price column)")
                continue
            
            prices = df[price_col].values
            
            if len(prices) != len(test_ids):
                print(f"⚠️  Skipping {filename} (shape mismatch)")
                continue
            
            historical_preds[filename] = prices
            scores[filename] = score
            print(f"✓ Loaded {filename}: SMAPE={score:.3f}%")
            
        except Exception as e:
            print(f"⚠️  Error loading {filename}: {str(e)}")
    
    if not historical_preds:
        print("\n⚠️  No valid historical predictions loaded")
        return None, None, None
    
    # Intelligent weighting
    print(f"\n🧠 Computing intelligent weights...")
    best_score = min(scores.values())
    
    weights = {}
    for filename, score in scores.items():
        score_diff = score - best_score
        weight = np.exp(-score_diff / 3.0)  # Exponential decay
        weights[filename] = weight
    
    # Normalize
    total_weight = sum(weights.values())
    weights = {k: v/total_weight for k, v in weights.items()}
    
    print(f"\n📊 Weight Distribution:")
    for name, weight in sorted(weights.items(), key=lambda x: -x[1]):
        score = scores[name]
        bar = '█' * int(weight * 50)
        print(f"   {name[:25]:25} | {weight:6.1%} {bar} (SMAPE: {score:.2f}%)")
    
    return historical_preds, weights, best_score

# ==================== MAIN PIPELINE ====================
def main():
    print("="*80)
    print("🚀 AUTOGLUON + ULTRA-RICH FEATURE ENGINEERING")
    print("="*80)
    print(f"⏱️  AutoGluon time limit: {CONFIG['time_limit']}s")
    print(f"🎯 Target SMAPE: <48%")
    print("="*80)
    
    # Load data
    print("\n[1/5] Loading data...")
    train_df = pd.read_csv(CONFIG['train_csv'])
    test_df = pd.read_csv(CONFIG['test_csv'])
    
    print(f"✓ Train: {len(train_df):,} samples")
    print(f"✓ Test: {len(test_df):,} samples")
    print(f"✓ Price range: ${train_df['price'].min():.2f} - ${train_df['price'].max():.2f}")
    
    # Load historical predictions
    print("\n[2/5] Loading historical predictions...")
    historical_preds, historical_weights, best_hist_score = load_historical_with_intelligence(test_df['sample_id'])
    
    # Feature engineering
    print("\n[3/5] Ultra-rich feature engineering...")
    train_features = create_ultra_rich_features(train_df)
    test_features = create_ultra_rich_features(test_df)
    
    # Prepare training data
    train_data = train_features.copy()
    train_data['price'] = train_df['price']
    test_data = test_features.copy()
    
    print(f"✓ Training shape: {train_data.shape}")
    print(f"✓ Test shape: {test_data.shape}")
    
    # Train AutoGluon
    print("\n[4/5] Training AutoGluon with rich features...")
    print("="*60)
    print("🎯 With 50+ engineered features, AutoGluon should perform much better!")
    print("="*60)
    
    predictor = TabularPredictor(
        label='price',
        problem_type='regression',
        eval_metric='mean_absolute_error',
        path=CONFIG['autogluon_path'],
        verbosity=2
    )
    
    predictor.fit(
        train_data,
        time_limit=CONFIG['time_limit'],
        presets=CONFIG['preset'],
        num_bag_folds=CONFIG['num_bag_folds'],
        num_bag_sets=1,
        num_stack_levels=2,  # Deeper stacking with rich features
        hyperparameters={
            'GBM': [
                {'extra_trees': True, 'ag_args': {'name_suffix': 'XT'}},
                {},
                {'learning_rate': 0.01, 'num_leaves': 256},
                {'learning_rate': 0.05, 'num_leaves': 128},
            ],
            'CAT': [
                {},
                {'iterations': 1000, 'learning_rate': 0.05},
            ],
            'XGB': [
                {},
                {'n_estimators': 500, 'learning_rate': 0.05},
            ],
            'RF': [
                {'n_estimators': 500, 'max_depth': 15},
                {'n_estimators': 300, 'max_depth': 20},
            ],
            'XT': [
                {'n_estimators': 500, 'max_depth': 15},
            ],
            'NN_TORCH': [{}],  # Neural network for complex patterns
        },
        keep_only_best=True,
        save_space=True,
    )
    
    print("\n✅ AutoGluon training complete!")
    
    # Show results
    leaderboard = predictor.leaderboard(train_data, silent=True)
    print("\n📊 MODEL LEADERBOARD")
    print("="*50)
    print(leaderboard.head(8)[['model', 'score_val', 'pred_time_val']])
    
    # Validation performance
    train_preds = predictor.predict(train_data.drop('price', axis=1))
    train_smape = calculate_smape(train_data['price'].values, train_preds)
    print(f"\n📈 AutoGluon OOF SMAPE: {train_smape:.4f}%")
    
    # Feature importance
    try:
        feature_importance = predictor.feature_importance(train_data, silent=True)
        print("\n🔍 TOP 10 FEATURE IMPORTANCE")
        print("="*40)
        print(feature_importance.head(10))
    except:
        print("Feature importance not available.")
    
    # Generate predictions
    print("\n[5/5] Generating predictions with meta-layer...")
    
    autogluon_preds = predictor.predict(test_data)
    autogluon_preds = np.maximum(autogluon_preds, 0.1)
    
    print(f"AutoGluon predictions:")
    print(f"  Mean: ${np.mean(autogluon_preds):.2f}")
    print(f"  Range: ${np.min(autogluon_preds):.2f} - ${np.max(autogluon_preds):.2f}")
    
    # Apply meta-layer if available
    if historical_preds is not None:
        print(f"\n🧠 Applying meta-layer...")
        
        historical_blend = np.zeros(len(test_data))
        for filename, preds in historical_preds.items():
            weight = historical_weights[filename]
            historical_blend += preds * weight
        
        # Intelligent blending based on OOF performance
        if train_smape < best_hist_score:
            alpha = 0.8  # Trust AutoGluon more
            expected_smape = train_smape * 0.98
            confidence = "HIGH"
        else:
            alpha = 0.6  # More conservative
            expected_smape = min(train_smape, best_hist_score) * 0.99
            confidence = "MEDIUM"
        
        final_predictions = alpha * autogluon_preds + (1 - alpha) * historical_blend
        
        print(f"📊 Meta-blending: {alpha:.0%} AutoGluon / {1-alpha:.0%} Historical")
        print(f"🎯 Expected SMAPE: ~{expected_smape:.2f}% ({confidence} confidence)")
    else:
        final_predictions = autogluon_preds
        expected_smape = train_smape + 0.5  # Conservative estimate
        print(f"🎯 Expected SMAPE: ~{expected_smape:.2f}%")
    
    # Final processing
    final_predictions = np.clip(final_predictions, 0.1, 10000)
    
    # Save submission
    submission = pd.DataFrame({
        'sample_id': test_df['sample_id'],
        'price': final_predictions
    })
    
    submission.to_csv(CONFIG['output_path'], index=False)
    
    print(f"\n" + "="*80)
    print("🎉 ULTRA-RICH FEATURE PIPELINE COMPLETE!")
    print("="*80)
    print(f"✓ Saved: {CONFIG['output_path']}")
    print(f"✓ Features used: {train_features.shape[1]}")
    print(f"🎯 Expected SMAPE: ~{expected_smape:.2f}%")
    print(f"📈 Improvement vs. basic (51.4%): ~{51.4 - expected_smape:.1f} percentage points")
    
    if expected_smape < 48:
        print("🏆 TARGET ACHIEVED: Expected SMAPE < 48%!")
    else:
        print("⚠️  May need more time or additional features for <48% SMAPE")
    
    return submission

if __name__ == "__main__":
    submission = main()
    print("\n✅ Pipeline completed successfully!")

In [ ]:
#KISHROE THIS OLD CODE
"""
AUTOGLUON + META-LEARNING ENSEMBLE
========================================================================
Strategy: Let AutoGluon handle model training, add intelligent meta-layer
Benefits: 
- AutoGluon trains 15+ models automatically
- Meta-layer learns from historical submissions
- Fast training (no manual CatBoost/XGBoost tuning)
Target: <48% SMAPE
========================================================================
"""

import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')
import glob
import os
from autogluon.tabular import TabularPredictor

# ==================== CONFIGURATION ====================
CONFIG = {
    'train_csv': '/kaggle/input/ml-challenge-2-2025/train.csv',
    'test_csv': '/kaggle/input/ml-challenge-2-2025/test.csv',
    'previous_outputs_dir': '/kaggle/input/previous-outputs',
    'output_path': '/kaggle/working/test_out.csv',
    'autogluon_path': './autogluon_models',
    
    # AutoGluon settings
    'time_limit': 3600,  # 1 hour (adjust based on Kaggle limits)
    'preset': 'best_quality',  # 'medium_quality', 'good_quality', or 'best_quality'
    'num_bag_folds': 7,  # More folds = better stability
    
    'random_state': 42,
    
    # Historical scores for meta-learning
    'historical_scores': {
        'output_49.385.csv': 49.38488936456298,
        'output_50.314.csv': 50.31381129428071,
        'output_51.375.csv': 51.3752876230568,
        'output_60.531.csv': 60.53146486141669,
        'output_61.586.csv': 61.58578507849365,
    }
}

# ==================== SMAPE METRIC ====================
def calculate_smape(actual, predicted):
    """Symmetric Mean Absolute Percentage Error"""
    actual = np.array(actual)
    predicted = np.array(predicted)
    predicted = np.maximum(predicted, 0.01)
    
    numerator = np.abs(predicted - actual)
    denominator = (np.abs(actual) + np.abs(predicted)) / 2
    smape = np.mean(numerator / (denominator + 1e-10)) * 100
    
    return smape

# ==================== ENHANCED FEATURE ENGINEERING ====================
def extract_pack_quantity(text):
    """Extract pack quantity with multiple patterns"""
    if pd.isna(text):
        return 1
    text_str = str(text).lower()
    
    # Value field (highest priority)
    value_match = re.search(r'value:\s*(\d+\.?\d*)', text_str)
    if value_match:
        val = float(value_match.group(1))
        if val > 1:
            return val
    
    # Pack patterns
    patterns = [
        r'\(pack of (\d+)\)', r'pack of (\d+)', r'\((\d+) pack\)',
        r'(\d+) pack', r'(\d+)-pack', r'(\d+)\s*count',
        r'set of (\d+)', r'(\d+)\s*piece', r'(\d+)\s*ct'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text_str)
        if match:
            qty = float(match.group(1))
            if 1 < qty < 1000:
                return qty
    
    return 1

def extract_unit_size(text):
    """Extract unit size and normalize to oz"""
    if pd.isna(text):
        return 0
    
    text_str = str(text).lower()
    
    patterns = [
        (r'(\d+\.?\d*)\s*ounce', 1),
        (r'(\d+\.?\d*)\s*oz', 1),
        (r'(\d+\.?\d*)\s*fl oz', 1),
        (r'(\d+\.?\d*)\s*pound', 16),
        (r'(\d+\.?\d*)\s*lb', 16),
        (r'(\d+\.?\d*)\s*ml', 0.033814),
        (r'(\d+\.?\d*)\s*liter', 33.814),
        (r'(\d+\.?\d*)\s*gram', 0.035274),
        (r'(\d+\.?\d*)\s*kg', 35.274),
    ]
    
    for pattern, conversion in patterns:
        match = re.search(pattern, text_str)
        if match:
            size = float(match.group(1))
            return size * conversion
    
    return 0

def extract_all_numbers(text):
    """Extract all numbers from text"""
    if pd.isna(text):
        return []
    numbers = re.findall(r'\d+\.?\d*', str(text))
    return [float(x) for x in numbers if float(x) < 100000]

def create_rich_features(df):
    """Create comprehensive feature set for AutoGluon"""
    print("🔧 Engineering features...")
    text = df['catalog_content'].fillna('')
    text_lower = text.str.lower()
    
    features = pd.DataFrame()
    
    # === PACK & SIZE FEATURES ===
    features['pack_qty'] = text.apply(extract_pack_quantity)
    features['unit_size_oz'] = text.apply(extract_unit_size)
    features['total_volume'] = features['pack_qty'] * features['unit_size_oz']
    features['is_multi_pack'] = (features['pack_qty'] > 1).astype(int)
    
    # Transformed versions
    features['log_pack_qty'] = np.log1p(features['pack_qty'])
    features['log_volume'] = np.log1p(features['total_volume'])
    features['sqrt_volume'] = np.sqrt(features['total_volume'])
    
    # === TEXT STATISTICS ===
    features['text_length'] = text.str.len()
    features['word_count'] = text.str.split().str.len()
    features['avg_word_len'] = features['text_length'] / (features['word_count'] + 1)
    features['bullet_count'] = text.str.count('Bullet Point')
    features['newline_count'] = text.str.count('\n')
    features['uppercase_ratio'] = text.str.count(r'[A-Z]') / (features['text_length'] + 1)
    
    # === NUMBER FEATURES ===
    all_numbers = text.apply(extract_all_numbers)
    features['num_count'] = all_numbers.apply(len)
    features['max_number'] = all_numbers.apply(lambda x: max(x) if x else 0)
    features['sum_numbers'] = all_numbers.apply(lambda x: sum(x) if x else 0)
    features['mean_numbers'] = all_numbers.apply(lambda x: np.mean(x) if x else 0)
    features['std_numbers'] = all_numbers.apply(lambda x: np.std(x) if len(x) > 1 else 0)
    
    # === QUALITY INDICATORS ===
    features['has_organic'] = text_lower.str.contains('organic', na=False).astype(int)
    features['has_premium'] = text_lower.str.contains('premium|gourmet|luxury', na=False).astype(int)
    features['has_gluten_free'] = text_lower.str.contains('gluten.free', na=False).astype(int)
    features['has_non_gmo'] = text_lower.str.contains('non.gmo', na=False).astype(int)
    features['has_vegan'] = text_lower.str.contains('vegan|plant.based', na=False).astype(int)
    features['has_kosher'] = text_lower.str.contains('kosher', na=False).astype(int)
    features['has_natural'] = text_lower.str.contains('natural|all.natural', na=False).astype(int)
    
    features['quality_score'] = (
        features['has_organic'] + features['has_premium'] + 
        features['has_gluten_free'] + features['has_non_gmo'] + 
        features['has_vegan'] + features['has_kosher'] + features['has_natural']
    )
    
    # === BRAND INDICATORS ===
    features['has_amazon'] = text_lower.str.contains('amazon|amazonbasics', na=False).astype(int)
    features['has_kirkland'] = text_lower.str.contains('kirkland', na=False).astype(int)
    features['has_top_brand'] = text_lower.str.contains(
        'starbucks|nescafe|keurig|folgers|nike|adidas|sony|samsung|apple', na=False
    ).astype(int)
    
    # === PRICE INDICATORS ===
    features['has_value'] = text_lower.str.contains('value|affordable|budget|economy', na=False).astype(int)
    features['has_luxury'] = text_lower.str.contains('luxury|premium|gourmet|artisan', na=False).astype(int)
    features['has_sale'] = text_lower.str.contains('discount|sale|deal|save', na=False).astype(int)
    
    # === CATEGORY INDICATORS ===
    features['is_food'] = text_lower.str.contains('food|snack|meal|eat|edible', na=False).astype(int)
    features['is_beverage'] = text_lower.str.contains('drink|beverage|juice|water|coffee', na=False).astype(int)
    features['is_electronics'] = text_lower.str.contains('electronic|digital|battery|usb|cable', na=False).astype(int)
    features['is_health'] = text_lower.str.contains('health|vitamin|supplement|medical', na=False).astype(int)
    features['is_beauty'] = text_lower.str.contains('beauty|cosmetic|skin|hair|makeup', na=False).astype(int)
    
    # === INTERACTION FEATURES (CRITICAL!) ===
    features['pack_x_size'] = features['pack_qty'] * features['unit_size_oz']
    features['pack_x_quality'] = features['pack_qty'] * features['quality_score']
    features['volume_x_quality'] = features['total_volume'] * features['quality_score']
    features['size_per_pack'] = features['unit_size_oz'] / (features['pack_qty'] + 1)
    features['quality_density'] = features['quality_score'] / (features['total_volume'] + 1)
    
    # === RATIO FEATURES ===
    features['numbers_per_word'] = features['num_count'] / (features['word_count'] + 1)
    features['bullets_per_100_chars'] = features['bullet_count'] / (features['text_length'] / 100 + 1)
    
    # === BINNED FEATURES ===
    features['volume_bin'] = pd.cut(
        features['total_volume'], 
        bins=[0, 5, 20, 50, 200, float('inf')], 
        labels=[0, 1, 2, 3, 4]
    ).astype(float)
    
    features['pack_bin'] = pd.cut(
        features['pack_qty'], 
        bins=[0, 1, 3, 6, 12, float('inf')], 
        labels=[0, 1, 2, 3, 4]
    ).astype(float)
    
    # Keep original text for AutoGluon's NLP models
    features['catalog_content'] = text
    
    features = features.fillna(0)
    print(f"✓ Created {features.shape[1]} features (including text)")
    
    return features

# ==================== LOAD HISTORICAL PREDICTIONS ====================
def load_historical_with_intelligence(test_ids):
    """Intelligently load and weight historical predictions"""
    print("\n" + "="*80)
    print("📚 LOADING HISTORICAL PREDICTIONS (META-LAYER)")
    print("="*80)
    
    if CONFIG['previous_outputs_dir'] is None or not os.path.exists(CONFIG['previous_outputs_dir']):
        print("ℹ️  No historical data directory found")
        print("   Training fresh AutoGluon model without meta-layer")
        return None, None, None
    
    csv_files = glob.glob(os.path.join(CONFIG['previous_outputs_dir'], '*.csv'))
    
    if not csv_files:
        print("ℹ️  No CSV files in historical directory")
        return None, None, None
    
    print(f"Found {len(csv_files)} historical submission(s)")
    
    historical_preds = {}
    scores = {}
    
    # Extract scores and load predictions
    for csv_file in csv_files:
        filename = os.path.basename(csv_file)
        
        # Try to extract SMAPE from filename
        score_match = re.search(r'(\d+\.\d+)', filename)
        if score_match:
            score = float(score_match.group(1))
        else:
            # Check configured scores
            score = None
            for key, val in CONFIG['historical_scores'].items():
                if key in filename or filename in key:
                    score = val
                    break
        
        if score is None:
            print(f"⚠️  Skipping {filename} (can't determine SMAPE)")
            continue
        
        try:
            df = pd.read_csv(csv_file)
            
            # Find price column flexibly
            price_col = None
            for col in df.columns:
                if 'price' in col.lower() or 'pred' in col.lower() or 'target' in col.lower():
                    price_col = col
                    break
            
            if price_col is None and len(df.columns) >= 2:
                price_col = df.columns[1]
            
            if price_col is None:
                print(f"⚠️  Skipping {filename} (no price column)")
                continue
            
            prices = df[price_col].values
            
            if len(prices) != len(test_ids):
                print(f"⚠️  Skipping {filename} (shape mismatch: {len(prices)} vs {len(test_ids)})")
                continue
            
            historical_preds[filename] = prices
            scores[filename] = score
            
            print(f"✓ Loaded {filename}: SMAPE={score:.3f}%")
            
        except Exception as e:
            print(f"⚠️  Error loading {filename}: {str(e)}")
    
    if not historical_preds:
        print("\n⚠️  No valid historical predictions loaded")
        return None, None, None
    
    # === INTELLIGENT WEIGHTING STRATEGY ===
    print(f"\n🧠 Computing intelligent weights...")
    
    # Strategy: Exponential weighting favoring better models
    # Better models get exponentially higher weight
    weights = {}
    best_score = min(scores.values())
    
    for filename, score in scores.items():
        # Exponential decay based on how far from best
        score_diff = score - best_score
        weight = np.exp(-score_diff / 5.0)  # Decay factor of 5
        weights[filename] = weight
    
    # Normalize weights
    total_weight = sum(weights.values())
    weights = {k: v/total_weight for k, v in weights.items()}
    
    # Show weight distribution
    print(f"\n📊 Weight Distribution (exponential decay):")
    sorted_items = sorted(weights.items(), key=lambda x: -x[1])
    for name, weight in sorted_items:
        score = scores[name]
        bar = '█' * int(weight * 50)
        print(f"   {name[:30]:30} | {weight:6.1%} {bar} (SMAPE: {score:.2f}%)")
    
    print(f"\n✨ Meta-layer ready with {len(historical_preds)} historical models")
    print(f"   Best historical SMAPE: {best_score:.3f}%")
    
    return historical_preds, weights, best_score

# ==================== MAIN PIPELINE ====================
def main():
    print("="*80)
    print("🚀 AUTOGLUON + META-LEARNING ENSEMBLE")
    print("="*80)
    print(f"⏱️  AutoGluon time limit: {CONFIG['time_limit']}s ({CONFIG['time_limit']//60} min)")
    print(f"🎯 Preset: {CONFIG['preset']}")
    print(f"📁 Bag folds: {CONFIG['num_bag_folds']}")
    print("="*80)
    
    # ==================== LOAD DATA ====================
    print("\n[1/5] Loading data...")
    train_df = pd.read_csv(CONFIG['train_csv'])
    test_df = pd.read_csv(CONFIG['test_csv'])
    
    train_df['catalog_content'] = train_df['catalog_content'].fillna('')
    test_df['catalog_content'] = test_df['catalog_content'].fillna('')
    
    print(f"✓ Train: {len(train_df):,} samples")
    print(f"✓ Test: {len(test_df):,} samples")
    print(f"✓ Price range: ${train_df['price'].min():.2f} - ${train_df['price'].max():.2f}")
    print(f"✓ Mean price: ${train_df['price'].mean():.2f}")
    
    # ==================== LOAD HISTORICAL ====================
    print("\n[2/5] Loading historical predictions...")
    historical_preds, historical_weights, best_hist_score = load_historical_with_intelligence(test_df['sample_id'])
    
    # ==================== FEATURE ENGINEERING ====================
    print("\n[3/5] Feature engineering...")
    train_features = create_rich_features(train_df)
    test_features = create_rich_features(test_df)
    
    # Prepare data for AutoGluon
    train_data = train_features.copy()
    train_data['price'] = train_df['price']
    
    test_data = test_features.copy()
    test_ids = test_df['sample_id']
    
    print(f"✓ Training shape: {train_data.shape}")
    print(f"✓ Test shape: {test_data.shape}")
    
    # ==================== TRAIN AUTOGLUON ====================
    print("\n[4/5] Training AutoGluon (this will take time)...")
    print("="*80)
    print("AutoGluon will automatically:")
    print("  - Train 15+ models (LightGBM, XGBoost, CatBoost, RF, NN, etc.)")
    print("  - Perform hyperparameter tuning")
    print("  - Create stacked ensembles")
    print("  - Select best combination")
    print("\n☕ Grab a coffee... this is where the magic happens!")
    print("="*80)
    
    predictor = TabularPredictor(
        label='price',
        problem_type='regression',
        eval_metric='mean_absolute_error',
        path=CONFIG['autogluon_path'],
        verbosity=2
    )
    
    predictor.fit(
        train_data,
        time_limit=CONFIG['time_limit'],
        presets=CONFIG['preset'],
        num_bag_folds=CONFIG['num_bag_folds'],
        num_bag_sets=1,
        num_stack_levels=1,
        hyperparameters={
            'GBM': [
                {'extra_trees': True, 'ag_args': {'name_suffix': 'XT'}},
                {},
                {'learning_rate': 0.01, 'num_leaves': 128},
            ],
            'CAT': {},
            'XGB': {},
            'RF': {'n_estimators': 300, 'max_depth': 15},
            'XT': {'n_estimators': 300, 'max_depth': 15},
        },
        keep_only_best=True,
        save_space=True,
    )
    
    print("\n✅ AutoGluon training complete!")
    
    # Show leaderboard
    print("\n" + "="*80)
    print("📊 MODEL LEADERBOARD")
    print("="*80)
    leaderboard = predictor.leaderboard(train_data, silent=True)
    print(leaderboard.head(10)[['model', 'score_val', 'pred_time_val']])
    
    # Validation performance
    train_preds = predictor.predict(train_data.drop('price', axis=1))
    train_smape = calculate_smape(train_data['price'].values, train_preds)
    
    print(f"\n📈 AutoGluon OOF SMAPE: {train_smape:.4f}%")
    
    # ==================== PREDICT + META-LAYER ====================
    print("\n[5/5] Generating predictions with meta-layer...")
    print("="*80)
    
    # Get AutoGluon predictions
    autogluon_preds = predictor.predict(test_data)
    autogluon_preds = np.maximum(autogluon_preds, 0.1)  # Ensure positive
    
    print(f"AutoGluon predictions:")
    print(f"  Mean: ${np.mean(autogluon_preds):.2f}")
    print(f"  Median: ${np.median(autogluon_preds):.2f}")
    print(f"  Range: ${np.min(autogluon_preds):.2f} - ${np.max(autogluon_preds):.2f}")
    
    # === APPLY META-LAYER ===
    if historical_preds is not None and len(historical_preds) > 0:
        print(f"\n🧠 Applying meta-layer blending...")
        
        # Create weighted historical blend
        historical_blend = np.zeros(len(test_data))
        for filename, preds in historical_preds.items():
            weight = historical_weights[filename]
            historical_blend += preds * weight
        
        print(f"Historical blend:")
        print(f"  Mean: ${np.mean(historical_blend):.2f}")
        print(f"  Median: ${np.median(historical_blend):.2f}")
        
        # === INTELLIGENT BLENDING STRATEGY ===
        # If AutoGluon OOF is better than best historical, trust it more
        if train_smape < best_hist_score:
            alpha = 0.75  # 75% AutoGluon, 25% historical
            confidence = "HIGH"
            expected_smape = train_smape * 0.98  # Expect slight improvement
        elif train_smape < best_hist_score + 1.0:
            alpha = 0.65  # 65% AutoGluon, 35% historical
            confidence = "MEDIUM"
            expected_smape = min(train_smape, best_hist_score) * 0.99
        else:
            alpha = 0.55  # 55% AutoGluon, 45% historical (conservative)
            confidence = "MEDIUM-LOW"
            expected_smape = best_hist_score * 0.98
        
        final_predictions = alpha * autogluon_preds + (1 - alpha) * historical_blend
        
        print(f"\n📊 Meta-Layer Blending:")
        print(f"  AutoGluon OOF:       {train_smape:.3f}%")
        print(f"  Best Historical:     {best_hist_score:.3f}%")
        print(f"  Blend ratio:         {alpha:.0%} AutoGluon / {1-alpha:.0%} Historical")
        print(f"  Confidence:          {confidence}")
        print(f"\n🎯 Expected SMAPE:    ~{expected_smape:.2f}% (±0.5%)")
        
    else:
        final_predictions = autogluon_preds
        expected_smape = train_smape + 1.0  # Account for generalization gap
        print(f"\nℹ️  No meta-layer (no historical data)")
        print(f"🎯 Expected SMAPE: ~{expected_smape:.2f}% (±1.0%)")
    
    # Final constraints
    final_predictions = np.clip(final_predictions, 0.1, 50000)
    
    # ==================== SAVE SUBMISSION ====================
    print("\n" + "="*80)
    print("💾 SAVING SUBMISSION")
    print("="*80)
    
    submission = pd.DataFrame({
        'sample_id': test_ids,
        'price': final_predictions
    })
    
    submission.to_csv(CONFIG['output_path'], index=False)
    
    print(f"✓ Saved: {CONFIG['output_path']}")
    print(f"✓ Samples: {len(submission):,}")
    print(f"✓ Final stats:")
    print(f"    Mean:   ${submission['price'].mean():.2f}")
    print(f"    Median: ${submission['price'].median():.2f}")
    print(f"    Range:  ${submission['price'].min():.2f} - ${submission['price'].max():.2f}")
    
    print("\n" + "="*80)
    print("🎉 COMPLETE!")
    print("="*80)
    print(f"🎯 Expected SMAPE: ~{expected_smape:.2f}%")
    
    if historical_preds:
        print(f"📚 Learned from {len(historical_preds)} previous submissions")
    
    print("\n📋 Next steps:")
    print("   1. Submit to leaderboard")
    print("   2. Record actual SMAPE")
    print("   3. Rename as: output_{actual_smape}.csv")
    print("   4. Add to historical dataset")
    print("   5. Re-run for iterative improvement! 🚀")
    print("="*80)
    
    return submission

if __name__ == "__main__":
    submission = main()
    print("\n✅ Script completed successfully!")

In [ ]:
from IPython.display import FileLink
import pandas as pd

# Check output
output_df = pd.read_csv('/kaggle/working/test_out.csv')
print(f"Output shape: {output_df.shape}")
print(f"Columns: {output_df.columns.tolist()}")
print(f"\nFirst 10 predictions:")
print(output_df.head(10))

# Statistics
print(f"\nPrice statistics:")
print(output_df['price'].describe())

# Download link
FileLink('/kaggle/working/test_out.csv')